# Target Sanity Check — Heatmap Generation

**Notebook role:** Phase 3 verification. Generates heatmap targets from real ground-truth (x, y) and visualises them to confirm the targets look right *before* we wire them into a training loop.

**What we check**
1. Single-frame heatmap with 1 person, 2, 3, 4
2. Empty room (frame from w22 / w23) -> all zeros
3. Round-trip: xy -> heatmap -> peaks -> xy
4. A short animation across consecutive frames to confirm temporal smoothness

**Outputs**
- `other_files/figs_eda/targets_*.png` -- figures used in the report


## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import Normalize

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.grid"] = False

ROOT = Path("..").resolve()
OTHER_FILES = ROOT / "other_files"
sys.path.insert(0, str(OTHER_FILES))

from heatmap import (
    ROOM_W, ROOM_H, GRID_W, GRID_H, CELL_W, CELL_H, SIGMA_CELLS,
    xy_to_cell, cell_to_xy,
    xy_to_heatmap, heatmap_to_xy,
    count_target, make_frame_stack, frame_stack_view,
)

DATA_DIR    = ROOT / "data" / "multi-person-localization"
WINDOWS_DIR = DATA_DIR / "data"
FIG_DIR     = OTHER_FILES / "figs_eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

window_files = sorted(WINDOWS_DIR.glob("window_*.npz"))
print(f"Grid: {GRID_W} x {GRID_H} cells of {CELL_W:.2f} x {CELL_H:.2f} m")
print(f"Sigma: {SIGMA_CELLS} cells = {SIGMA_CELLS*CELL_W*100:.0f} cm")


## 1. Single-frame heatmaps for each occupancy level

Pick one frame from each occupancy class and visualise the target heatmap with the ground-truth markers overlaid.


In [ ]:
def plot_heatmap_panel(ax, h, xy, mask, title=""):
    """Plot one heatmap with GT markers overlaid.

    The heatmap is drawn with origin='lower' so y increases upward,
    matching the room's coordinate system.
    """
    # extent: maps cell index space to metric space
    extent = (0, ROOM_W, 0, ROOM_H)
    ax.imshow(h, origin="lower", extent=extent, cmap="hot",
              vmin=0, vmax=1, aspect="equal", interpolation="nearest")
    # Room outline
    ax.add_patch(patches.Rectangle((0, 0), ROOM_W, ROOM_H,
                                   fill=False, edgecolor="white", linewidth=1))
    # Ground-truth markers
    valid = mask.astype(bool)
    if valid.any():
        ax.scatter(xy[valid, 0], xy[valid, 1], marker="o",
                   s=80, facecolors="none", edgecolors="cyan", linewidths=1.5)
    ax.set_xlim(-0.1, ROOM_W + 0.1)
    ax.set_ylim(-0.1, ROOM_H + 0.1)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x (m)", fontsize=8)
    ax.set_ylabel("y (m)", fontsize=8)
    ax.tick_params(labelsize=7)


# Pick representative windows: 0p, 1p, 2p, 3p, 4p
# From metadata: 22/23 = 0p; 16-21 = 1p; 0-4 = 2p; 12-15 = 3p; 5-11 = 4p
chosen = [
    ("0 people", 22, None),
    ("1 person",  18, None),
    ("2 people",  0,  None),
    ("3 people",  12, None),
    ("4 people",  5,  None),
]

# Pick the same approximate timestamp (frame ~3000 = 120s in)
fixed_t = 3000

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
for ax, (title, w_idx, _) in zip(axes, chosen):
    p = window_files[w_idx]
    with np.load(p) as f:
        xy   = f["people_xy"][fixed_t]      # (4, 2)
        mask = f["people_mask"][fixed_t]    # (4,)
    h = xy_to_heatmap(xy, mask)
    plot_heatmap_panel(ax, h, xy, mask, title=f"{title} (w{w_idx:02d}, t={fixed_t})")

plt.suptitle("Heatmap targets across occupancy levels", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "targets_per_occupancy.png", bbox_inches="tight")
plt.show()


## 2. Round-trip on real frames

Take 50 frames from a 4-person window, generate the heatmap, run the peak extractor, and compare to ground truth.


In [ ]:
W_IDX = 5     # 4-people random walk
N_FRAMES = 50
FRAME_OFFSET = 1000

with np.load(window_files[W_IDX]) as f:
    xy_gt   = f["people_xy"][FRAME_OFFSET:FRAME_OFFSET + N_FRAMES]
    mask_gt = f["people_mask"][FRAME_OFFSET:FRAME_OFFSET + N_FRAMES]

# Generate heatmaps
heatmaps = xy_to_heatmap(xy_gt, mask_gt)
counts_gt = count_target(mask_gt)
print(f"Heatmap stack shape: {heatmaps.shape}")
print(f"Counts: {dict(zip(*np.unique(counts_gt, return_counts=True)))}")

# Round-trip: extract peaks using the GT count
errors = []
for t in range(N_FRAMES):
    k = int(counts_gt[t])
    pred = heatmap_to_xy(heatmaps[t], k=k)
    if k == 0:
        assert pred.shape == (0, 2)
        continue
    # Greedy match: for each GT person, nearest pred peak
    gt_xy = xy_gt[t][mask_gt[t].astype(bool)]
    used = set()
    for true_xy in gt_xy:
        dists = np.linalg.norm(pred - true_xy, axis=1)
        for j in np.argsort(dists):
            if int(j) not in used:
                used.add(int(j))
                errors.append(dists[j])
                break

errors = np.array(errors)
print(f"\nRound-trip per-person error over {N_FRAMES} frames:")
print(f"  mean   = {errors.mean()*100:.2f} cm")
print(f"  median = {np.median(errors)*100:.2f} cm")
print(f"  max    = {errors.max()*100:.2f} cm")
print(f"  90th   = {np.percentile(errors, 90)*100:.2f} cm")


In [ ]:
# Visual round-trip check on the most error-prone frame
worst_idx_in_errs = int(np.argmax(errors))
# Find which frame that error came from
cum = np.cumsum(counts_gt)
worst_t = int(np.searchsorted(cum, worst_idx_in_errs + 1))
print(f"Worst error frame: t={worst_t}, error={errors[worst_idx_in_errs]*100:.2f} cm")

# Plot it
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

xy_t   = xy_gt[worst_t]
mask_t = mask_gt[worst_t]
k_t    = int(counts_gt[worst_t])
h_t    = heatmaps[worst_t]
pred_t = heatmap_to_xy(h_t, k=k_t)

# Panel 1: heatmap with GT
plot_heatmap_panel(axes[0], h_t, xy_t, mask_t,
                   title=f"Target heatmap (w{W_IDX:02d}, t={worst_t}, k={k_t})")

# Panel 2: heatmap with predictions
extent = (0, ROOM_W, 0, ROOM_H)
axes[1].imshow(h_t, origin="lower", extent=extent, cmap="hot",
               vmin=0, vmax=1, aspect="equal", interpolation="nearest")
axes[1].add_patch(patches.Rectangle((0, 0), ROOM_W, ROOM_H,
                                    fill=False, edgecolor="white", linewidth=1))
valid = mask_t.astype(bool)
axes[1].scatter(xy_t[valid, 0], xy_t[valid, 1], marker="o", s=80,
                facecolors="none", edgecolors="cyan", linewidths=1.5, label="GT")
if len(pred_t):
    axes[1].scatter(pred_t[:, 0], pred_t[:, 1], marker="x", s=80,
                    color="lime", linewidths=2, label="Predicted peak")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_xlim(-0.1, ROOM_W + 0.1); axes[1].set_ylim(-0.1, ROOM_H + 0.1)
axes[1].set_title("With extracted peaks")
axes[1].set_xlabel("x (m)"); axes[1].set_ylabel("y (m)")

# Panel 3: error distribution
axes[2].hist(errors * 100, bins=30, color="steelblue", edgecolor="black")
axes[2].axvline(errors.mean() * 100, color="red", linestyle="--",
                label=f"mean = {errors.mean()*100:.1f} cm")
axes[2].set_xlabel("Round-trip error (cm)")
axes[2].set_ylabel("Count")
axes[2].set_title(f"Error distribution ({N_FRAMES} frames, w{W_IDX:02d})")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "targets_round_trip.png", bbox_inches="tight")
plt.show()


## 3. Temporal smoothness

Plot the heatmap for 6 consecutive frames during a walking sequence. The peaks should drift smoothly between cells, not jump.


In [ ]:
W_IDX = 5
START = 1500
STEP = 4   # show every 4th frame -> 4/25 = 0.16s apart

with np.load(window_files[W_IDX]) as f:
    xy_seq   = f["people_xy"][START:START + 6*STEP:STEP]
    mask_seq = f["people_mask"][START:START + 6*STEP:STEP]

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
for i, ax in enumerate(axes):
    h = xy_to_heatmap(xy_seq[i], mask_seq[i])
    plot_heatmap_panel(ax, h, xy_seq[i], mask_seq[i],
                       title=f"t = {(START + i*STEP)/25:.2f} s")

plt.suptitle(f"Temporal evolution of target heatmap (w{W_IDX:02d}, every {STEP}/25 = {STEP*40} ms)",
             y=1.02, fontsize=11)
plt.tight_layout()
plt.savefig(FIG_DIR / "targets_temporal.png", bbox_inches="tight")
plt.show()


## 4. Frame stack sanity check

Verify the temporal stacking matches what the model will receive: at frame t, the model sees frames [t-7, t-6, ..., t] arranged on a leading axis.


In [ ]:
# Use a small dummy CIR-like array
T_test = 20
T_CTX = 8
dummy = np.arange(T_test * 6 * 3 * 105 * 2, dtype=np.float32).reshape(T_test, 6, 3, 105, 2)

stack = make_frame_stack(dummy, T_context=T_CTX)
print(f"Input shape: {dummy.shape}")
print(f"Stack shape: {stack.shape}  (T, T_context, R, A, B, 2)")

# Check causality: stack[10] should equal dummy[3:11]
np.testing.assert_array_equal(stack[10], dummy[3:11])
print("\nCausality check OK: stack[10] == dummy[3:11]")

# Check warmup behaviour: stack[0] should be 8 copies of dummy[0]
np.testing.assert_array_equal(stack[0, 0], dummy[0])
np.testing.assert_array_equal(stack[0, 7], dummy[0])
print("Warmup check OK: stack[0] is 8 copies of dummy[0]")

# Per-frame memory cost
per_frame_kb = stack.shape[1] * stack.shape[2] * stack.shape[3] * stack.shape[4] * stack.shape[5] * 4 / 1024
print(f"\nPer-frame stacked input size: {per_frame_kb:.2f} KB ({per_frame_kb*1024:.0f} float32 elements)")
print(f"  -- this is what the model sees per inference call")


## 5. Summary

If the figures above all look right, we're done with target generation. The targets are:
- Heatmap: shape (24, 36), values in [0, 1], peak = 1.0 at the cell containing each person
- Count: integer in {0, 1, 2, 3, 4}, used both as a one-hot training target and as `k` in inference

Next phase: **model architecture** -- writing the Keras spec for the encoder + heatmap decoder + count head, all using only TFLM-supported ops.
